# 15 Safety


## Setup


In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / 'scripts'))

import numpy as np
import pandas as pd

import analysis
from analysis import (CELL_ORDER, CONTROLS, CUE_DIRECTION, FOCUS,
                      IMPLICIT_ADULT, IMPLICIT_MINOR, MACRO, NEUTRAL, ORDER,
                      Register, SIGNAL, STATED, STATED_ADULT, STATED_MINOR,
                      STRATA, THRESHOLD_CONTRAST, TRAJECTORY,
                      bootstrap_paired, by_scenario, contrast,
                      fingerprint_line, interval, leave_one_out, load_corpus,
                      present, publish, pvalue, rate_by_model,
                      stratified_two_sample, trend_by_scenario,
                      write_captions)

pd.set_option('display.width', 220, 'display.max_columns', 40,
              'display.max_colwidth', 64)

REGISTER = Register('safety')


In [ ]:
frame = load_corpus()
returned = frame[frame['responded']]

focus = returned[returned['scenario_type'].eq(FOCUS)]
harmful = returned[returned['scenario_type'].eq('Harmful')]

# Validation


print(f'source      {frame.attrs["source"]}')
print(f'corpus      {fingerprint_line(frame)}')
# Family Size
assert frame.attrs['testable'] == analysis.TESTABLE, (
    f'the characteristics clearing the agreement floor have changed: '
    f'{frame.attrs["testable"]} against the declared {analysis.TESTABLE}. The '
    f'response characteristics family is sized on the declared set.')
print(f'testable    {len(frame.attrs["testable"])} characteristics at or above '
      f'kappa {analysis.KAPPA_FLOOR:.2f}, matching the declared set')
print(f'prevalence  {", ".join(frame.attrs["untestable"])} below the floor, '
      f'reported but never tested')


## Outcomes


In [ ]:
CELL_COLUMNS = {cell: cell.lower().replace(' ', '_') for cell in CELL_ORDER}

# Blocked Replies




outcome = pd.DataFrame({cell: rate_by_model(returned, column)
                        for cell, column in CELL_COLUMNS.items()})
outcome['Refusal Rate'] = rate_by_model(returned, 'refusal')
outcome['Content Delivery Rate'] = rate_by_model(returned, 'delivered')
outcome['Decision-Delivery Mismatch'] = rate_by_model(returned, 'divergence')
outcome.loc[MACRO] = outcome.mean()
outcome = outcome.round(1)
outcome.index.name = 'Model'

publish(outcome, 'safety_s01_outcome')


In [ ]:
# Export





NAMES = {'refusal': 'Refusal Rate', 'delivered': 'Content Delivery Rate',
         'divergence': 'Decision-Delivery Mismatch'}
by_stratum = pd.DataFrame([
    {'Scenario type': stratum, 'Measure': NAMES[measure],
     **rate_by_model(returned[returned['scenario_type'].eq(stratum)], measure)
     .round(1).to_dict()}
    for stratum in STRATA for measure in NAMES])

publish(by_stratum, 'safety_s02_outcome_by_stratum')


## Model Comparison


In [ ]:
STRATUM_ROWS = [
    ('Benign', returned['scenario_type'].eq('Benign')),
    ('Rights', returned['scenario_type'].eq('Rights')),
    ('Age Restricted, Explicit Age (Minor)',
     returned['scenario_type'].eq(FOCUS) & returned['condition'].isin(STATED_MINOR)),
    ('Age Restricted, Explicit Age (Adult)',
     returned['scenario_type'].eq(FOCUS) & returned['condition'].isin(STATED_ADULT)),
    ('Harmful', returned['scenario_type'].eq('Harmful')),
]


# Sampling




def alignment_column(part, measure):
    per_model = {label: by_scenario(part[part['label'] == label], measure) * 100
                 for label in ORDER}
    column = {label: round(float(series.mean()), 1)
              for label, series in per_model.items()}
    point, low, high = analysis.macro_average(per_model)
    column[MACRO] = round(point, 1)
    column['95% CI Lower'] = round(low, 1)
    column['95% CI Upper'] = round(high, 1)
    return column


# Model Names

alignment = pd.DataFrame({name: alignment_column(returned[mask], 'action_aligned')
                          for name, mask in STRATUM_ROWS})
alignment.index.name = 'Model'

publish(alignment.reset_index(), 'safety_02_alignment')


In [ ]:
# Decision Alignment


components = []
for name, mask in STRATUM_ROWS:
    part = returned[mask]
    for measure, label in (('decision_aligned', 'Decision Alignment'),
                           ('action_aligned', 'Benchmark Action Alignment')):
        # Blocked Replies
        
        
        
        components.append({'Benchmark condition': name, 'Measure': label,
                           **rate_by_model(part, measure).round(1).to_dict()})
# Model Alignment
components = pd.DataFrame(components).melt(
    id_vars=['Benchmark condition', 'Measure'], var_name='Model',
    value_name='value')
components = (components.pivot(index=['Model', 'Benchmark condition'],
                               columns='Measure', values='value')
              .reindex(ORDER, level='Model').reset_index())
components.columns.name = None
components['Difference (pp)'] = (components['Decision Alignment']
                          - components['Benchmark Action Alignment']).round(1)

publish(components, 'safety_s03_alignment_components')


In [ ]:
# Withheld Replies
























DIRECTIONAL = [
    ('A. Content Leakage', 'Harmful Content Delivered',
     frame['scenario_type'].eq('Harmful'), 'delivered'),
    ('A. Content Leakage', 'Age Restricted Content Delivered, Explicit Age (Minor)',
     frame['scenario_type'].eq(FOCUS) & frame['condition'].isin(STATED_MINOR),
     'delivered'),
    ('B. Over-Restriction', 'Age Restricted Request Refused, Explicit Age (Adult)',
     frame['scenario_type'].eq(FOCUS) & frame['condition'].isin(STATED_ADULT),
     'refusal'),
    ('B. Over-Restriction', 'Benign or Rights Request Refused',
     frame['scenario_type'].isin(['Rights', 'Benign']), 'refusal'),
]

conditional, end_to_end = [], []
for panel, name, mask, measure in DIRECTIONAL:
    submitted = frame[mask]
    answered = submitted[submitted['responded']]
    # Conditional Fields
    first = rate_by_model(answered, measure).round(1).to_dict()
    # Blocked Replies
    withheld = 0.0 if measure == 'delivered' else 1.0
    submitted = submitted.assign(
        reached=submitted[measure].fillna(0.0) + submitted['blocked'] * withheld)
    second = rate_by_model(submitted, 'reached').round(1).to_dict()
    for store, row in ((conditional, first), (end_to_end, second)):
        store.append({'Panel': panel, 'Directional rate': name, **row,
                      MACRO: round(float(np.mean(list(row.values()))), 1)})

directional = pd.DataFrame(conditional).melt(
    id_vars=['Panel', 'Directional rate'], var_name='Model', value_name='Rate (%)')
directional = directional[directional['Model'].isin(ORDER + [MACRO])]
publish(directional, 'safety_s04_directional')


In [ ]:
sensitivity = pd.DataFrame(end_to_end).melt(
    id_vars=['Panel', 'Directional rate'], var_name='Model', value_name='Rate (%)')
sensitivity = sensitivity[sensitivity['Model'].isin(ORDER + [MACRO])]
sensitivity = sensitivity.merge(
    directional.rename(columns={'Rate (%)': 'Response-conditional (%)'}),
    on=['Panel', 'Directional rate', 'Model'])
sensitivity['Shift (pp)'] = (sensitivity['Rate (%)']
                             - sensitivity['Response-conditional (%)']).round(1)

publish(sensitivity, 'safety_s05_directional_end_to_end')


## Contrasts


In [ ]:
PRIMARY = [(TRAJECTORY, STATED_MINOR, STATED_ADULT),
           (THRESHOLD_CONTRAST, ['age17'], ['age18']),
           (SIGNAL, STATED_MINOR, IMPLICIT_MINOR)]
effects = {name: contrast(focus, 'refusal', first, second,
                          'age conditioning', name, REGISTER)
           for name, first, second in PRIMARY}

for stratum in CONTROLS:
    contrast(returned[returned['scenario_type'].eq(stratum)], 'refusal',
             STATED_MINOR, STATED_ADULT, 'benchmark control',
             f'{TRAJECTORY} on {stratum}', REGISTER)

# Contrast

CUES = [(CUE_DIRECTION, IMPLICIT_MINOR, IMPLICIT_ADULT)]
for name, first, second in CUES:
    contrast(focus, 'refusal', first, second, 'implicit cue', name, REGISTER)

CATEGORY = 'Instruction against Information, within harm domain'
stratified_two_sample(harmful, 'refusal', 'category', 'Instruction',
                      'Information', 'domain', 'prompt category', CATEGORY,
                      REGISTER)

for column in frame.attrs['testable']:
    contrast(returned, column, STATED_MINOR, STATED_ADULT,
             'response characteristics',
             f'{column}, {TRAJECTORY}', REGISTER)

FLAT = {}
for label in ORDER:
    rho, flat, p = trend_by_scenario(focus[focus['label'] == label], 'refusal')
    point, low, high = bootstrap_paired(rho)
    FLAT[label] = flat
    REGISTER.add('age trend', 'Mean Scenario-Level Rank Trend in the Explicit Age',
                 label, point, low, high, p, n=int(rho.size), measure='refusal')

register = analysis.adjust(REGISTER.frame())
print(register.groupby(['tier', 'family']).agg(
    tests=('p', 'size'), tested=('p', 'count'),
    significant=('significant', 'sum')).to_string())


## Primary Contrasts


In [ ]:
primary = pd.concat([present(register, name).assign(Contrast=name)
                     for name, _, _ in PRIMARY])
primary = primary[['Contrast', 'Model'] + ['Effect (pp)', 'p', 'q', '95% CI Lower', '95% CI Upper', 'Scenarios']]

publish(primary, 'safety_01_primary')


In [ ]:
levels = pd.DataFrame({
    name: rate_by_model(focus, 'refusal', group) for name, group in (
        ('Control (No Age)', [NEUTRAL]),
        ('Implicit Cue (Adult)', IMPLICIT_ADULT),
        ('Explicit Age (Adult)', STATED_ADULT),
        ('Implicit Cue (Minor)', IMPLICIT_MINOR),
        ('Explicit Age (Minor)', STATED_MINOR))}).round(1)
levels.loc[MACRO] = levels.mean().round(1)
levels.index.name = 'Model'

publish(levels, 'safety_s06_signal_levels')


In [ ]:
# Blocked Replies


restricted_cells = (frame[frame['scenario_type'].eq(FOCUS)]
                    .groupby(['label', 'scenario_id', 'condition'])['responded']
                    .sum())
retention = pd.DataFrame({
    'Age Restricted scenarios paired': (
        present(register, PRIMARY[0][0]).set_index('Model')['Scenarios']),
    'Blocked requests, Age Restricted': (
        frame[frame['scenario_type'].eq(FOCUS)].groupby('label')['blocked'].sum()),
    'Cells with no returned replicate': (
        restricted_cells.eq(0).groupby('label').sum()),
    'Cells with some but not all replicates': (
        restricted_cells.between(1, 2).groupby('label').sum()),
}).reindex(ORDER).astype('Int64')
retention.index.name = 'Model'

publish(retention, 'safety_s07_retention')


## Planned Controls


In [ ]:
controls = pd.concat([
    present(register, f'{TRAJECTORY} on {stratum}')
    .assign(Stratum=stratum) for stratum in CONTROLS])
controls = controls[['Stratum', 'Model'] + ['Effect (pp)', 'p', 'q', '95% CI Lower', '95% CI Upper', 'Scenarios']]

publish(controls, 'safety_s08_controls')


## Implicit Cues


In [ ]:
cues = present(register, CUE_DIRECTION)

publish(cues, 'safety_s09_cues')


## Request Category


In [ ]:
category = present(register, CATEGORY).rename(columns={'Scenarios': 'Domains'})
# Scenario Weighting


levels_by_category = pd.DataFrame(
    {name: rate_by_model(harmful[harmful['category'].eq(name)], 'refusal')
     for name in ('Instruction', 'Information')}).round(1)
levels_by_category.loc[MACRO] = levels_by_category.mean().round(1)
category = category.merge(
    levels_by_category.rename(columns={'Instruction': 'Instruction (%)',
                                       'Information': 'Information (%)'})
    .rename_axis('Model').reset_index(), on='Model', how='left')

publish(category, 'safety_s10_category')


## Harm Domains


In [ ]:
domains = pd.DataFrame(
    {domain: rate_by_model(part, 'refusal')
     for domain, part in harmful.groupby('domain')}).T.round(1)
domains = domains.reindex(columns=ORDER)
domains.index.name = 'domain'
domains[MACRO] = domains.mean(axis=1).round(1)
domains['Scenarios'] = harmful.groupby('domain')['scenario_id'].nunique()
domains = domains.sort_values(MACRO, ascending=False)

publish(domains, 'safety_s11_domains')


## Response Characteristics


In [ ]:
rows = []
for column, group in frame.attrs['group_of'].items():
    tested = column in frame.attrs['testable']
    name = f'{column}, {TRAJECTORY}'
    part = register[register['contrast'] == name].set_index('model') \
        if tested else None
    for label in ORDER:
        rows.append({
            'Group': group,
            'Characteristic': column.replace('_', ' ').title(),
            'Model': label,
            'Prevalence (%)': round(float(
                by_scenario(returned[returned['label'] == label], column).mean()
                * 100), 2),
            'Effect (pp)': analysis.bounds(
                part.at[label, 'effect'], part.at[label, 'low'],
                part.at[label, 'high'], sign=True)['estimate'] if tested else '',
            '95% CI Lower': analysis.bounds(
                part.at[label, 'effect'], part.at[label, 'low'],
                part.at[label, 'high'])['low'] if tested else '',
            '95% CI Upper': analysis.bounds(
                part.at[label, 'effect'], part.at[label, 'low'],
                part.at[label, 'high'])['high'] if tested else '',
            'q': pvalue(part.at[label, 'q']) if tested else 'not tested',
        })

characteristics = pd.DataFrame(rows)
publish(characteristics, 'safety_s12_characteristics')
characteristics.head(12)


### Replicate Stability


In [ ]:
# Replicates


returned = returned.assign(main_response=(returned['answer'] == 'Refusal').astype(float))
stability = pd.DataFrame([
    {'Characteristic': column.replace('_', ' ').title(),
     'Cells': result['cells'],
     'Unanimous (%)': round(result['unanimous'] * 100, 1),
     'Active Cells': result['active'],
     'Unanimous Among Active (%)': round(result['unanimous_active'] * 100, 1)}
    for column, result in ((name, analysis.stability(returned, name))
                           for name in ['main_response'] + analysis.RUBRIC)])

publish(stability, 'safety_s17_stability')


### Characteristic Decomposition


In [ ]:
# Effect





# Contrast

delivery = []
for name, label, first, second in [
        (PRIMARY[0][0], TRAJECTORY, STATED_MINOR, STATED_ADULT),
        (PRIMARY[1][0], THRESHOLD_CONTRAST, ['age17'], ['age18']),
        (PRIMARY[2][0], SIGNAL, STATED_MINOR, IMPLICIT_MINOR)]:
    for model in ORDER:
        panel = focus[focus['label'] == model]
        refused = analysis.differences(panel, 'refusal', first, second) * 100
        arrived = analysis.differences(panel, 'delivered', first, second) * 100
        point, low, high = bootstrap_paired(arrived)
        delivery.append({'Contrast': label, 'Model': model,
                         'Refusal Rate (pp)': round(float(refused.mean()), 1),
                         'Content Delivery (pp)': round(point, 1),
                         '95% CI Lower': round(low, 1),
                         '95% CI Upper': round(high, 1)})
delivery = pd.DataFrame(delivery)
publish(delivery, 'safety_s18_delivery')

# Effect

LEGAL_BLOCKS = [
    ('Benign', returned['scenario_type'].eq('Benign')),
    ('Rights', returned['scenario_type'].eq('Rights')),
    ('Age Restricted, Explicit Age (Minor)',
     returned['scenario_type'].eq(FOCUS) & returned['condition'].isin(STATED_MINOR)),
    ('Age Restricted, Explicit Age (Adult)',
     returned['scenario_type'].eq(FOCUS) & returned['condition'].isin(STATED_ADULT)),
    ('Harmful', returned['scenario_type'].eq('Harmful')),
]
levels = pd.DataFrame({name: rate_by_model(returned[mask], 'legal_statement')
                       for name, mask in LEGAL_BLOCKS}).round(1)
shifts = pd.DataFrame({
    stratum: {model: round(float(analysis.differences(
        returned[returned['scenario_type'].eq(stratum)
                 & returned['label'].eq(model)],
        'legal_statement', STATED_MINOR, STATED_ADULT).mean() * 100), 1)
        for model in ORDER}
    for stratum in STRATA})
legal = levels.join(shifts.add_suffix(', Minor Minus Adult (pp)'))
legal.index.name = 'Model'
publish(legal, 'safety_s19_legal_strata')

# Contrast



conditioned = []
for cell in ('Strong Refusal', 'Total Compliance'):
    part = returned[returned['outcome'].eq(cell)]
    for column in frame.attrs['testable']:
        for model in ORDER:
            diff = analysis.differences(part[part['label'] == model], column,
                                        STATED_MINOR, STATED_ADULT) * 100
            point, low, high = bootstrap_paired(diff)
            conditioned.append({
                'Outcome Cell': cell,
                'Characteristic': column.replace('_', ' ').title(),
                'Model': model, 'Effect (pp)': round(point, 1),
                '95% CI Lower': round(low, 1), '95% CI Upper': round(high, 1),
                'Scenarios': int(diff.size)})
publish(pd.DataFrame(conditioned), 'safety_s20_conditioned')

# Age Boundary



minors = []
for model in ORDER:
    panel = focus[focus['label'] == model]
    inside, flat, _ = trend_by_scenario(panel, 'refusal', STATED_MINOR)
    ladder, _, _ = trend_by_scenario(panel, 'refusal', STATED)
    point, low, high = bootstrap_paired(inside)
    minors.append({'Model': model,
                   'Rho, Ages 7 To 17': round(point, 3),
                   '95% CI Lower': round(low, 3), '95% CI Upper': round(high, 3),
                   'Rho, Ages 7 To 21': round(float(ladder.mean()), 3),
                   'Scenarios': int(inside.size), 'Flat': flat})
publish(pd.DataFrame(minors), 'safety_s22_trend_minors')

# Conditional Fields


eligible = returned[analysis.ALTERNATIVE].notna()
alternative = pd.DataFrame([
    {'Model': model,
     'Eligible Replies': int(eligible[returned['label'].eq(model)].sum()),
     'Eligible (%)': round(float(eligible[returned['label'].eq(model)].mean() * 100), 1),
     'Alternative Among Eligible (%)': round(float(by_scenario(
         returned[returned['label'] == model], analysis.ALTERNATIVE).mean() * 100), 2),
     # Scenario Weighting
     
     
     
     
     
     
     
     
     
     'Alternative Over All Replies (%)': round(float(by_scenario(
         returned[returned['label'] == model].assign(
             whole=returned.loc[returned['label'] == model,
                                analysis.ALTERNATIVE].fillna(0.0)),
         'whole').mean() * 100), 2)}
    for model in ORDER])
publish(alternative, 'safety_s21_alternative')


## Robustness


In [ ]:
summary = (register.groupby(['tier', 'family'])
           .agg(tests=('p', 'size'), tested=('p', 'count'),
                significant=('significant', 'sum'),
                smallest_q=('q', 'min'))
           .reset_index())
summary['tier'] = pd.Categorical(summary['tier'], analysis.TIERS, ordered=True)
summary = summary.sort_values(['tier', 'family']).round(4)

publish(summary, 'safety_s13_register')


In [ ]:
# Correlation


trend = present(register, 'Mean Scenario-Level Rank Trend in the Explicit Age',
                unit='rho', places=3, order=ORDER)
trend['Flat scenarios'] = trend['Model'].map(FLAT).astype('Int64')
publish(trend, 'safety_s14_trend')

names = [name for name, _, _ in PRIMARY]
labels = [TRAJECTORY, THRESHOLD_CONTRAST, SIGNAL]
withheld = pd.DataFrame({label: leave_one_out(effects[name])
                         for name, label in zip(names, labels)}).round(1)
withheld.index.name = 'Macro-Average With This Model Withheld'
withheld.loc['Reported Six Model Macro-Average'] = [
    round(float(effects[name].mean()), 1) for name in names]
withheld.loc['Models With the Majority Sign'] = [
    f'{int(max((effects[name] > 0).sum(), (effects[name] < 0).sum()))} of {len(ORDER)}'
    for name in names]

publish(withheld, 'safety_s15_sensitivity')


In [ ]:
# Agreement



family = register[register['family'].eq('response characteristics')
                  & register['p'].notna()]
withheld = family[~family['contrast'].str.startswith(tuple(analysis.BORDERLINE))]
requalified = analysis.benjamini_hochberg(withheld['p'])

borderline = pd.DataFrame([
    {'Family': 'Response Characteristics, as Declared',
     'Characteristics': family['contrast'].nunique(),
     'Tests': len(family),
     'Significant after correction': int((family['q'] < analysis.Q).sum())},
    {'Family': 'Response Characteristics, Borderline Withheld',
     'Characteristics': withheld['contrast'].nunique(),
     'Tests': len(withheld),
     'Significant after correction': int((requalified < analysis.Q).sum())},
])
# Tests Remain Many


changed = int(((family.loc[withheld.index, 'q'] < analysis.Q)
               != (requalified < analysis.Q)).sum())
borderline['Tests on other characteristics that change'] = ['', changed]

publish(borderline, 'safety_s16_borderline')


## Outputs


In [ ]:
path = REGISTER.write()
captions = write_captions()

print(f'{len(REGISTER.frame())} rows registered to '
      f'{path.relative_to(analysis.ROOT)}')
print(f'captions merged into {captions.relative_to(analysis.ROOT)}')
print(dict(analysis.WRITTEN))
